# Local GGUF Eval: v1 vs v2

This notebook re-runs local evaluation for both GGUF models on the same harness.

What it does:
- loads `v1_qwen3b-soap-q4_k_m.gguf` and `v2_qwen3b-multitemplate-q4_k_m.gguf`
- uses prompt-time schema injection for every template
- evaluates `soap`, `referral_a`, `referral_b`, and `mse`
- defaults to Apple Metal offload when available, otherwise CPU


In [1]:
import importlib.metadata as metadata
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path


def run_text(command):
    try:
        result = subprocess.run(
            command, capture_output=True, text=True, check=False
        )
        text = (result.stdout or result.stderr).strip()
        return text or "<no output>"
    except Exception as exc:
        return f"<unavailable: {exc}>"


print("Python:", sys.version)
print("Platform:", platform.platform())
print("Machine:", platform.machine())
print("CPU cores:", os.cpu_count())

if sys.platform == "darwin":
    mem_bytes = run_text(["sysctl", "-n", "hw.memsize"])
    if mem_bytes.isdigit():
        print("Unified memory (GB):", round(int(mem_bytes) / (1024 ** 3), 2))
    print("\nMetal / GPU info:")
    gpu_lines = run_text(["system_profiler", "SPDisplaysDataType"]).splitlines()
    print("\n".join(gpu_lines[:20]))
elif shutil.which("nvidia-smi"):
    print("\nGPU info:")
    print(run_text(["nvidia-smi"]))

print("\nllama-cpp-python:", metadata.version("llama-cpp-python"))

from llama_cpp import Llama, llama_cpp as lib

GPU_OFFLOAD_SUPPORTED = bool(lib.llama_supports_gpu_offload())
AUTO_N_GPU_LAYERS = -1 if GPU_OFFLOAD_SUPPORTED else 0

print("GPU offload supported:", GPU_OFFLOAD_SUPPORTED)
print("Default n_gpu_layers:", AUTO_N_GPU_LAYERS)


Python: 3.12.12 (main, Dec  9 2025, 19:05:33) [Clang 21.1.4 ]
Platform: macOS-15.7.4-arm64-arm-64bit
Machine: arm64
CPU cores: 10
Unified memory (GB): 16.0

Metal / GPU info:


ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library


Graphics/Displays:

    Apple M2 Pro:

      Chipset Model: Apple M2 Pro
      Type: GPU
      Bus: Built-In
      Total Number of Cores: 16
      Vendor: Apple (0x106b)
      Metal Support: Metal 3
      Displays:
        Color LCD:
          Display Type: Built-in Liquid Retina XDR Display
          Resolution: 3024 x 1964 Retina
          Main Display: Yes
          Mirror: Off
          Online: Yes
          Automatically Adjust Brightness: Yes
          Connection Type: Internal
        PHL 271B8Q:

llama-cpp-python: 0.3.21


ggml_metal_library_init: loaded in 10.097 sec
ggml_metal_rsets_init: creating a residency set collection (keep_alive = 180 s)
ggml_metal_device_init: GPU name:   MTL0 (Apple M2 Pro)
ggml_metal_device_init: GPU family: MTLGPUFamilyApple8  (1008)
ggml_metal_device_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_device_init: GPU family: MTLGPUFamilyMetal3  (5001)
ggml_metal_device_init: simdgroup reduction   = true
ggml_metal_device_init: simdgroup matrix mul. = true
ggml_metal_device_init: has unified memory    = true
ggml_metal_device_init: has bfloat            = true
ggml_metal_device_init: has tensor            = false
ggml_metal_device_init: use residency sets    = true
ggml_metal_device_init: use shared buffers    = true
ggml_metal_device_init: recommendedMaxWorkingSetSize  = 11453.25 MB


GPU offload supported: True
Default n_gpu_layers: -1


In [2]:
import json
import re
import shutil
import time
from collections import defaultdict
from functools import lru_cache

from src.data_generation.templates import REGISTRY
from src.data_generation.validate import (
    _check_mse,
    _check_referral,
    _check_soap,
    _collect_spans,
)
from src.prompts import build_inference_messages


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "models").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root from current working directory")


REPO_ROOT = find_repo_root(Path.cwd())
MODELS = {
    "v1": REPO_ROOT / "models" / "v1_qwen3b-soap-q4_k_m.gguf",
    "v2": REPO_ROOT / "models" / "v2_qwen3b-multitemplate-q4_k_m.gguf",
}
EVAL_FILES = {
    "soap": REPO_ROOT / "data" / "qwen3.5_latest" / "eval_in_dist.soap.jsonl",
    "referral_a": REPO_ROOT / "data" / "qwen3.5_latest" / "eval_in_dist.referral_a.jsonl",
    "referral_b": REPO_ROOT / "data" / "qwen3.5_latest" / "eval_zero_shot.referral_b.jsonl",
    "mse": REPO_ROOT / "data" / "qwen3.5_latest" / "eval_zero_shot.mse.jsonl",
}

N_CTX = 4096
MAX_TOKENS = 512
N_THREADS = max(1, (os.cpu_count() or 4) - 2)
N_GPU_LAYERS = AUTO_N_GPU_LAYERS  # set to 0 to force CPU
SHOW_EXAMPLES = 1

for path in [*MODELS.values(), *EVAL_FILES.values()]:
    assert path.exists(), f"Missing required file: {path}"

print("Repo root:", REPO_ROOT)
print("Models:")
for name, path in MODELS.items():
    print(f"  {name}: {path.name} ({path.stat().st_size / 1e9:.2f} GB)")
print("n_ctx:", N_CTX)
print("max_tokens:", MAX_TOKENS)
print("n_threads:", N_THREADS)
print("n_gpu_layers:", N_GPU_LAYERS)


Repo root: /Users/nizamuddin/clinical_transcript_summariser
Models:
  v1: v1_qwen3b-soap-q4_k_m.gguf (1.93 GB)
  v2: v2_qwen3b-multitemplate-q4_k_m.gguf (1.93 GB)
n_ctx: 4096
max_tokens: 512
n_threads: 8
n_gpu_layers: -1


In [3]:
def load_jsonl(path: Path):
    with path.open() as f:
        return [json.loads(line) for line in f if line.strip()]


def print_table(rows, columns=None, digits=4):
    if not rows:
        print("No rows")
        return
    if columns is None:
        columns = list(rows[0].keys())

    def fmt(value):
        if isinstance(value, float):
            return f"{value:.{digits}f}"
        return str(value)

    widths = {}
    for col in columns:
        widths[col] = max(len(col), max(len(fmt(row.get(col, ""))) for row in rows))

    header = " | ".join(col.ljust(widths[col]) for col in columns)
    divider = "-+-".join("-" * widths[col] for col in columns)
    print(header)
    print(divider)
    for row in rows:
        print(" | ".join(fmt(row.get(col, "")).ljust(widths[col]) for col in columns))


DATA = {template: load_jsonl(path) for template, path in EVAL_FILES.items()}

print_table(
    [
        {
            "template": template,
            "rows": len(rows),
            "label_key": REGISTRY[template]["label_key"],
            "file": str(EVAL_FILES[template].relative_to(REPO_ROOT)),
        }
        for template, rows in DATA.items()
    ],
    columns=["template", "rows", "label_key", "file"],
)


template   | rows | label_key | file                                               
-----------+------+-----------+----------------------------------------------------
soap       | 10   | soap      | data/qwen3.5_latest/eval_in_dist.soap.jsonl        
referral_a | 10   | referral  | data/qwen3.5_latest/eval_in_dist.referral_a.jsonl  
referral_b | 10   | referral  | data/qwen3.5_latest/eval_zero_shot.referral_b.jsonl
mse        | 10   | mse       | data/qwen3.5_latest/eval_zero_shot.mse.jsonl       


In [4]:
_OBJECT_RE = re.compile(r"\{.*\}", re.DOTALL)
_FENCE_RE = re.compile(r"^```(?:json)?\s*\n?(.*?)\n?```$", re.DOTALL)

EXPECTED_TOP_KEYS = {
    "soap": {"subjective", "objective", "assessment", "plan"},
    "referral_a": {"specialty", "patient", "reason", "history", "current_meds", "request"},
    "referral_b": {"specialty", "patient", "reason", "history", "current_meds", "request"},
    "mse": {"appearance", "behaviour", "speech", "mood", "affect", "thought", "cognition", "insight"},
}
SCHEMA_CHECKERS = {
    "soap": _check_soap,
    "referral_a": _check_referral,
    "referral_b": _check_referral,
    "mse": _check_mse,
}


def unwrap_json_object(raw: str) -> str:
    raw = raw.strip()
    fence = _FENCE_RE.match(raw)
    if fence:
        raw = fence.group(1).strip()
    match = _OBJECT_RE.search(raw)
    return match.group(0) if match else raw


def parse_prediction(raw: str):
    try:
        return json.loads(unwrap_json_object(raw))
    except Exception:
        return None


def collect_content_values(node):
    values = []
    if isinstance(node, dict):
        if {"text", "evidence_span"}.issubset(node.keys()):
            values.append(node.get("text"))
        elif {"name", "evidence_span"}.issubset(node.keys()):
            values.append(node.get("name"))
        elif {"action", "evidence_span"}.issubset(node.keys()):
            values.append(node.get("action"))
        else:
            for value in node.values():
                values.extend(collect_content_values(value))
    elif isinstance(node, list):
        for item in node:
            values.extend(collect_content_values(item))
    return values


def content_token_set(node):
    tokens = set()
    for value in collect_content_values(node):
        if isinstance(value, str) and value.strip():
            tokens.update(value.lower().split())
    return tokens


def content_jaccard(pred_label, gold_label):
    pred_tokens = content_token_set(pred_label)
    gold_tokens = content_token_set(gold_label)
    if not pred_tokens and not gold_tokens:
        return 1.0
    return len(pred_tokens & gold_tokens) / max(len(pred_tokens | gold_tokens), 1)


def non_null_content_rate(label):
    values = collect_content_values(label)
    if not values:
        return 0.0
    filled = sum(1 for value in values if isinstance(value, str) and value.strip())
    return filled / len(values)


def is_all_null_prediction(label):
    return all(not (isinstance(value, str) and value.strip()) for value in collect_content_values(label))


In [5]:
@lru_cache(maxsize=None)
def load_llm(model_name: str):
    model_path = MODELS[model_name]
    print(f"[load] {model_name}: {model_path.name}")
    t0 = time.perf_counter()
    llm = Llama(
        model_path=str(model_path),
        n_ctx=N_CTX,
        n_gpu_layers=N_GPU_LAYERS,
        n_threads=N_THREADS,
        verbose=False,
        seed=0,
    )
    print(f"[load] ready in {time.perf_counter() - t0:.1f}s")
    return llm


def generate(model_name: str, template_name: str, transcript: str) -> str:
    llm = load_llm(model_name)
    messages = build_inference_messages(template_name, transcript)
    response = llm.create_chat_completion(
        messages=messages,
        temperature=0.0,
        max_tokens=MAX_TOKENS,
    )
    return response["choices"][0]["message"]["content"].strip()


def score_prediction(template_name: str, pred_raw: str, gold_label: dict, transcript: str):
    pred_label = parse_prediction(pred_raw)
    if pred_label is None or not isinstance(pred_label, dict):
        return {
            "parse": 0,
            "top_keys": 0,
            "schema_valid": 0,
            "content_overlap": 0.0,
            "evidence_grounding": 0.0,
            "non_null_content_rate": 0.0,
            "all_null": 1,
            "pred_label": None,
        }

    schema_err = SCHEMA_CHECKERS[template_name](pred_label)
    spans = [span for span in _collect_spans(pred_label) if span]
    if not spans:
        grounding = 0.0
    else:
        grounding = sum(1 for span in spans if span in transcript) / len(spans)

    return {
        "parse": 1,
        "top_keys": int(set(pred_label.keys()) >= EXPECTED_TOP_KEYS[template_name]),
        "schema_valid": int(schema_err is None),
        "content_overlap": content_jaccard(pred_label, gold_label),
        "evidence_grounding": grounding,
        "non_null_content_rate": non_null_content_rate(pred_label),
        "all_null": int(is_all_null_prediction(pred_label)),
        "pred_label": pred_label,
    }


In [6]:
def summarise(model_name: str, template_name: str, details):
    n = len(details)
    return {
        "model": model_name,
        "template": template_name,
        "n": n,
        "json_parse_rate": sum(d["parse"] for d in details) / n,
        "top_level_keys_rate": sum(d["top_keys"] for d in details) / n,
        "schema_valid_rate": sum(d["schema_valid"] for d in details) / n,
        "content_overlap_mean": sum(d["content_overlap"] for d in details) / n,
        "evidence_grounding_rate": sum(d["evidence_grounding"] for d in details) / n,
        "non_null_content_rate_mean": sum(d["non_null_content_rate"] for d in details) / n,
        "all_null_rate": sum(d["all_null"] for d in details) / n,
        "mean_latency_ms": sum(d["latency_ms"] for d in details) / n,
    }


def evaluate_model(model_name: str):
    summaries = []
    all_details = []

    for template_name, rows in DATA.items():
        label_key = REGISTRY[template_name]["label_key"]
        template_details = []
        print(f"[run] {model_name} -> {template_name} ({len(rows)} rows)")

        for index, row in enumerate(rows, start=1):
            t0 = time.perf_counter()
            pred_raw = generate(model_name, template_name, row["transcript"])
            latency_ms = (time.perf_counter() - t0) * 1000
            scores = score_prediction(
                template_name, pred_raw, row[label_key], row["transcript"]
            )
            detail = {
                "model": model_name,
                "template": template_name,
                "index": index,
                "transcript": row["transcript"],
                "gold": row[label_key],
                "pred_raw": pred_raw,
                "pred_label": scores.pop("pred_label"),
                "latency_ms": latency_ms,
                **scores,
            }
            template_details.append(detail)
            all_details.append(detail)
            print(
                f"  [{index:>2}/{len(rows)}] {latency_ms:>7.0f} ms  "
                f"parse={detail['parse']} schema={detail['schema_valid']} "
                f"overlap={detail['content_overlap']:.2f} "
                f"ground={detail['evidence_grounding']:.2f} "
                f"all_null={detail['all_null']}"
            )

        summaries.append(summarise(model_name, template_name, template_details))

    summaries.append(summarise(model_name, "ALL", all_details))
    return summaries, all_details


In [7]:
v1_summaries, v1_details = evaluate_model("v1")
v2_summaries, v2_details = evaluate_model("v2")
all_summaries = v1_summaries + v2_summaries

print_table(
    all_summaries,
    columns=[
        "model",
        "template",
        "n",
        "json_parse_rate",
        "schema_valid_rate",
        "content_overlap_mean",
        "evidence_grounding_rate",
        "non_null_content_rate_mean",
        "all_null_rate",
        "mean_latency_ms",
    ],
    digits=4,
)


[run] v1 -> soap (10 rows)
[load] v1: v1_qwen3b-soap-q4_k_m.gguf


llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


[load] ready in 1.3s
  [ 1/10]    6784 ms  parse=1 schema=1 overlap=0.31 ground=0.83 all_null=0
  [ 2/10]    2831 ms  parse=1 schema=1 overlap=0.32 ground=1.00 all_null=0
  [ 3/10]    3886 ms  parse=1 schema=1 overlap=0.24 ground=1.00 all_null=0
  [ 4/10]    7584 ms  parse=1 schema=1 overlap=0.33 ground=0.90 all_null=0
  [ 5/10]    5309 ms  parse=1 schema=1 overlap=0.44 ground=1.00 all_null=0
  [ 6/10]    4088 ms  parse=1 schema=1 overlap=0.41 ground=1.00 all_null=0
  [ 7/10]    4231 ms  parse=1 schema=1 overlap=0.30 ground=1.00 all_null=0
  [ 8/10]    3213 ms  parse=1 schema=1 overlap=0.18 ground=0.75 all_null=0
  [ 9/10]    6114 ms  parse=1 schema=1 overlap=0.38 ground=1.00 all_null=0
  [10/10]    3376 ms  parse=1 schema=1 overlap=0.50 ground=1.00 all_null=0
[run] v1 -> referral_a (10 rows)
  [ 1/10]    4539 ms  parse=1 schema=1 overlap=0.51 ground=1.00 all_null=0
  [ 2/10]    2713 ms  parse=1 schema=1 overlap=0.55 ground=0.83 all_null=0
  [ 3/10]    3157 ms  parse=1 schema=1 overlap

llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


[load] ready in 1.5s
  [ 1/10]    8527 ms  parse=1 schema=0 overlap=0.55 ground=0.90 all_null=0
  [ 2/10]    2978 ms  parse=1 schema=1 overlap=0.17 ground=1.00 all_null=0
  [ 3/10]    4177 ms  parse=1 schema=1 overlap=0.55 ground=1.00 all_null=0
  [ 4/10]    8291 ms  parse=1 schema=1 overlap=0.41 ground=0.91 all_null=0
  [ 5/10]    5347 ms  parse=1 schema=1 overlap=0.59 ground=1.00 all_null=0
  [ 6/10]    4881 ms  parse=1 schema=1 overlap=0.57 ground=1.00 all_null=0
  [ 7/10]    6535 ms  parse=1 schema=1 overlap=0.55 ground=1.00 all_null=0
  [ 8/10]    3660 ms  parse=1 schema=1 overlap=0.26 ground=0.83 all_null=0
  [ 9/10]    8263 ms  parse=1 schema=1 overlap=0.57 ground=0.89 all_null=0
  [10/10]    3909 ms  parse=1 schema=1 overlap=0.81 ground=1.00 all_null=0
[run] v2 -> referral_a (10 rows)
  [ 1/10]    5176 ms  parse=1 schema=1 overlap=0.68 ground=0.88 all_null=0
  [ 2/10]    3289 ms  parse=1 schema=1 overlap=0.85 ground=1.00 all_null=0
  [ 3/10]    3065 ms  parse=1 schema=1 overlap

In [8]:
summary_lookup = {(row["model"], row["template"]): row for row in all_summaries}

comparison = []
for template_name in DATA:
    v1_row = summary_lookup[("v1", template_name)]
    v2_row = summary_lookup[("v2", template_name)]
    comparison.append(
        {
            "template": template_name,
            "v1_overlap": v1_row["content_overlap_mean"],
            "v2_overlap": v2_row["content_overlap_mean"],
            "delta_overlap": v2_row["content_overlap_mean"] - v1_row["content_overlap_mean"],
            "v1_all_null": v1_row["all_null_rate"],
            "v2_all_null": v2_row["all_null_rate"],
            "v1_ground": v1_row["evidence_grounding_rate"],
            "v2_ground": v2_row["evidence_grounding_rate"],
        }
    )

print_table(
    comparison,
    columns=[
        "template",
        "v1_overlap",
        "v2_overlap",
        "delta_overlap",
        "v1_all_null",
        "v2_all_null",
        "v1_ground",
        "v2_ground",
    ],
    digits=4,
)


def show_examples(details, model_name: str, template_name: str, limit: int = 1):
    shown = 0
    for detail in details:
        if detail["model"] != model_name or detail["template"] != template_name:
            continue
        print("=" * 80)
        print(f"MODEL: {model_name}  TEMPLATE: {template_name}  INDEX: {detail['index']}")
        print("TRANSCRIPT:")
        print(detail["transcript"])
        print("\nPREDICTED RAW:")
        print(detail["pred_raw"])
        print("\nGOLD:")
        print(json.dumps(detail["gold"], indent=2))
        shown += 1
        if shown >= limit:
            break


for template_name in DATA:
    show_examples(v1_details, "v1", template_name, limit=SHOW_EXAMPLES)
    show_examples(v2_details, "v2", template_name, limit=SHOW_EXAMPLES)


template   | v1_overlap | v2_overlap | delta_overlap | v1_all_null | v2_all_null | v1_ground | v2_ground
-----------+------------+------------+---------------+-------------+-------------+-----------+----------
soap       | 0.3404     | 0.5032     | 0.1628        | 0.0000      | 0.0000      | 0.9483    | 0.9531   
referral_a | 0.5037     | 0.7400     | 0.2363        | 0.0000      | 0.0000      | 0.8405    | 0.9875   
referral_b | 0.2651     | 0.4678     | 0.2028        | 0.0000      | 0.0000      | 0.8550    | 0.9121   
mse        | 0.0546     | 0.0104     | -0.0442       | 0.0000      | 0.6000      | 0.9100    | 0.4000   
MODEL: v1  TEMPLATE: soap  INDEX: 1
TRANSCRIPT:
HCP: hi there, whats going on
Patient: hello doctor, i have this nasty cough that wont go away
HCP: how long has it been going on
Patient: about a week now, started after i caught a cold
HCP: any wheezing or chest tightness
Patient: yeah, feels like i cant breathe properly when i exercise
HCP: ok, let me listen to your l

## Interpretation
This local GGUF comparison was designed to separate transfer after single-template SFT (`v1`) from transfer after multi-template SFT (`v2`) under the same prompt-time schema injection and local Metal inference setup.

### Main comparison
- v2 beats v1 on soap, referral_a, and referral_b.
- Deltas in content_overlap_mean:
  - soap: +0.1628
  - referral_a: +0.2363
  - referral_b: +0.2028
That is the core result:
- prompt-time schema injection plus v1 already gives some transfer
- but v2 is clearly better on the related template family
What that means
- v1 is not zero-shot hopeless on referral.
- So schema text plus SOAP-only SFT already carries some transfer.
- But v2 is better on both referral splits, so multi-template SFT is adding real value.
Referral interpretation
- v1
  - referral_a: 0.5037
  - referral_b: 0.2651
- v2
  - referral_a: 0.7400
  - referral_b: 0.4678
So:
- both models do worse on referral_b than referral_a
- that supports the existing story that referral_b is still a harder style-transfer setting even though the schema semantics are shared
MSE interpretation
This is the most important nuance.
Raw numbers:
- v1 mse
  - content_overlap_mean = 0.0546
  - schema_valid_rate = 1.0000
  - all_null_rate = 0.0000
- v2 mse
  - content_overlap_mean = 0.0104
  - schema_valid_rate = 0.0000
  - all_null_rate = 0.6000
At first glance, v1 looks better on MSE.
I would not treat that as true ontology success.
Why:
- v1 is producing schema-shaped guesses with copied spans.
- The qualitative example shows wrong field mapping, even when spans are grounded.
- v2 is failing more conservatively, often by abstaining / nulling out fields.
So the MSE result is better described as:
- v1: over-eager, schema-shaped guessing
- v2: conservative abstention

### Main findings
- `v2` outperformed `v1` on `soap`, `referral_a`, and `referral_b`.
- The largest gains were on referral:
  - `referral_a`: `0.5037 -> 0.7400`
  - `referral_b`: `0.2651 -> 0.4678`
- This means prompt-time schema injection plus SOAP-only SFT already carries some transfer, but multi-template SFT adds clear value on the related referral template family.
### How to interpret referral
- `referral_b` remains harder than `referral_a` for both models, despite shared referral schema semantics.
- This supports the existing interpretation that `referral_b` is a harder style-transfer case, not a fully new ontology test.
### How to interpret MSE
- Neither model solved zero-shot `mse`.
- `v1` produced schema-shaped outputs with copied grounded spans, but qualitative inspection shows poor field semantics.
- `v2` failed more conservatively, often abstaining with partial or null outputs.
- So the MSE comparison is better understood as:
  - `v1`: over-eager schema-shaped guessing
  - `v2`: conservative abstention
### Caveats
- `schema_valid_rate` in this notebook is stricter than the earlier report's top-level key check, so absolute MSE schema numbers are not directly comparable to the Kaggle report.
- `content_overlap_mean` is a generalized overlap metric for cross-template comparison and should be compared within template, not across different templates.
- `evidence_grounding_rate` can remain high even when content is placed in the wrong field.
### Takeaway
- `v1` is the correct baseline for asking how far single-template SFT plus schema injection can transfer.
- `v2` shows that multi-template SFT improves transfer on nearby template families.
- Neither `v1` nor `v2` demonstrates robust zero-shot transfer to a genuinely new clinical ontology like `mse`.